Problem Statement
A small electronics store wants a chatbot that can answer natural-language questions about product prices using an LLM with tool-calling, built without any third-party agent framework (no LangChain, no LangGraph). You will define a get_product_price tool and implement the manual reason-act loop to wire it up to an LLM.

Dataset
Use this in-memory product catalog:

CATALOG = {
    "Wireless Mouse": 799,
    "USB-C Charger": 1299,
    "Bluetooth Speaker": 2499,
    "Laptop Stand": 1099,
    "Mechanical Keyboard": 3499
}
Tasks
Write the JSON-style tool description for get_product_price (the format an LLM's tool/function-calling interface expects): include the tool's name, a clear description of what it does, and its parameters (name, type, and whether required).
Implement the Python function get_product_price(product_name) that looks up CATALOG and returns a structured result. If the product isn't found, return a structured error result instead of raising an exception or returning plain text.
Implement the manual agent loop (no agent framework) that: (a) initializes the message list with a system prompt and a user prompt, (b) calls the LLM with the messages and the tool description, (c) checks whether the LLM's output requests a tool call and, if so, extracts the tool name and parameters, (d) executes the matching Python function, (e) appends the tool's output back into the message history, and (f) repeats the call to the LLM with the updated messages until it returns a final answer instead of another tool request.
Trace your implementation for the user question "How much does a Wireless Mouse cost?" — show, in order, the tool call the LLM would request, the output your function returns, and the final natural-language answer the loop would produce.
Expected Output
Complete, runnable Python code for the tool description, the get_product_price function, and the manual loop, plus a written trace (as described in Task 4) for the sample query.

Submission Format
Paste your complete code and the written trace directly in the input box.

LLM call: Implement call_llm(messages, tools) as a deterministic mock function if needed— you do not need a real LLM API, provider, SDK, or credentials. Your mock should: on the first call (when the last message is the user's question), return a response that requests the get_product_price tool call with the product name extracted from the question; on the second call (after the tool's structured output has been appended to the message history), return a final natural-language answer built from that structured output. This keeps the question about the reason-act loop mechanics (extracting tool calls, invoking the function, appending results, terminating correctly) rather than about live LLM behavior.

Note: You may use any IDE (online: Google Colab, Jupyter Notebook; offline: VS Code, PyCharm, or similar) to write, run, and test your code. You must disable all AI-based extensions (GitHub Copilot, Tabnine, Codeium, etc.) before starting — this refers to IDE autocomplete/assistant tools (Copilot, Tabnine, Codeium, etc.) that would write the code for you, not to the mock call_llm function this question asks you to implement yourself. You may refer to official documentation (e.g., your LLM provider's API docs, Python standard library docs) while writing your code. Submit your complete code as a copy-pasted snippet in the input box — do not submit a URL, notebook link, GitHub repository, or Google Drive file.



In [6]:
from dotenv import load_dotenv
import os
from groq import Groq
import json

In [2]:
load_dotenv()
GroqApik=os.getenv("GroqAPIKey")

In [5]:
client = Groq(api_key=GroqApik)

### DEFINING TOOLS

In [ ]:
def get_product_price(productName:str) -> str:
    try:
        CATALOG = {
            "wireless mouse": 799,
            "usb-c charger": 1299,
            "bluetooth speaker": 2499,
            "laptop stand": 1099,
            "mechanical keyboard": 3499
            }
        price = CATALOG.get(productName.lower())
        if(price):
            return json.dumps(
                {
                    "status":"success",
                    "Product Name":productName,
                    "Price":price
                }
            )
             
        else:
            return json.dumps(
                {
                    "status":"fail",
                    "Product Name":productName,
                    "message":f"Sorry!! {productName} is not found in our catelogue."
                }
            )
    except Exception as e:
        return json.dumps(
                {
                    "status":"error",
                    "Product Name":productName,
                    "message":f"Exception in get_product_price: {e}"
                }
            )

In [9]:
print(get_product_price("Mouse"))
print(get_product_price("laptop stand"))

{"status": "fail", "Product Name": "Mouse", "message": "Sorry!! Mouse is not found in our catelogue."}
{"status": "success", "Product Name": "laptop stand", "Price": 1099}


### Declaring the tools for LLM's usage

In [11]:
electronicsTools = [
    {
        "type":"function",
        "function":{
            "name":"get_product_price",
            "description":"To get the product prices.",
            "parameters":{
                "type":"object",
                "properties":{
                    "productName":{
                        "type":"string",
                        "description":"The product for which the price has to be fetched."
                    }
                },
                "required":["productName"]
            }
        }
    }
]

In [12]:
availableTools={
    "get_product_price":get_product_price
}

### Agentic Loop definition

In [38]:
def electronicsChatbot(maxIterations,userMsg,verbose):
    alltoolResults=[]
    message = [
        {
            "role":"system",
            "content":"You are an useful assistant, which provided prices for the products available in the catelog or else with a message of product not found."
        },{
            "role":"user",
            "content":userMsg
        }
    ]

    if verbose:
        print(f"message to LLM: {message}")

    for step in range(maxIterations):
        response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=message,
            tools=electronicsTools
        )    

        choice = response.choices[0]

        if choice.finish_reason == 'stop':
            break

        if choice.message.tool_calls:
            message.append(choice.message)

            toolCalls = choice.message.tool_calls

            for tool in toolCalls:
                funcName = tool.function.name
                arguments = json.loads(tool.function.arguments)

                if verbose:
                    print(f"tool name: {funcName}, arguments: {arguments}")

                toolresult = availableTools[funcName](**arguments)

                message.append({
                    "role":"tool",
                    "tool_call_id": tool.id,
                    "content":toolresult
                })

                alltoolResults.append({
                    "toolName":funcName,
                    "arguments":arguments,
                    "toolResult":toolresult
                })

    finalResponse = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{
            "role":"system",
            "content":"Provide a response to the user question in a json format with tags Question, Answer- answer again a json with tags Product Name and Price. If price is not found, then display the message"
        },
        {
            "role":"user",
            "content":f"Data gathered from tools:{alltoolResults}. User Request: {userMsg}"
        }],
        response_format={"type":"json_object"}
    )

    if verbose:
        print(f"Final Response: {finalResponse}")

    responsereport = json.loads(finalResponse.choices[0].message.content)
    return responsereport

In [35]:
response = electronicsChatbot(maxIterations=5,userMsg="How much does a Wireless Mouse cost?",verbose=False)
print(f"response from LLM: \n{json.dumps(response,indent=2)}")

response from LLM: 
{
  "Question": "How much does a Wireless Mouse cost?",
  "Answer": {
    "Product Name": "Wireless Mouse",
    "Price": 799
  }
}


In [ ]:
response = electronicsChatbot(maxIterations=5,userMsg="What is the price of a Pen?",verbose=False)
print(f"response from LLM: \n{json.dumps(response,indent=2)}")

response from LLM: 
{
  "Question": "What is the price of a Pen?",
  "Answer": {
    "Product Name": "Pen",
    "Price": "Sorry!! Pen is not found in our catelogue."
  }
}
